# Iowa data centers and nitrate

## Reproducibility audit and observed-data workflow

The original nitrate and weather values were generated. Replicating those values does **not** validate a zero effect. This notebook bundles the actual EPA Iowa extract for 2012–2019, keeps it separate from forensic outputs, and does not estimate causal effects without a reviewed source/exposure crosswalk.

Run the cells in order in Google Colab. No ZIP upload is required. The optional public-source refresh downloads about 89 MiB. The bundled snapshot is sufficient for all default cells.


In [ ]:
import sys, subprocess, importlib.util
needed = [m for m in ['numpy','pandas','scipy','matplotlib'] if importlib.util.find_spec(m) is None]
if needed: subprocess.check_call([sys.executable, '-m', 'pip', 'install', *needed])


In [ ]:
from pathlib import Path
import base64, gzip, json, os
ROOT = Path.cwd() / 'Iowa_Nitrate_Audit'
ROOT.mkdir(exist_ok=True)
PAYLOAD = {'data/raw/epa_syr4_iowa_nitrate_raw.csv': '...', 'data/raw/dnr_current_system_index.csv': '...', 'data/required_inputs.json': 'H4sIAAAAAAACA31X244UNxB9z1dY+7wDu5BEIfuEWEVCChsEARRFqOWx3dNm3XbjywwdxL/nlN0X9+4CQojxpVx1qs6p6q8/MXY2JD+4oM5+Z2dvlHBeBubV56S9kmyvWucVUyHqnkdtD4yLmLhhkke+E8pG5ZlqWyViuGIieY+lvMkG5XsdGU+S/rWSSRWE10PUzjJnzXh2Tq9zy80YdGhwXjvZ0NN4Tkn48y8O4MiTi4tfdheX+Juv5JUnv+4unu2eXpxh4WMxdOTa8L1RMGEUzwEtBi6f3DVw+WyHxaeXqwGro+dRNYH3g1EB17+Ww8ElLzI+L92Js+ubNzmekIbBjIRJitroqBUgcPvItWUWYB2B25fB+RjYSceuoCK1IAC4H2dXWq2MDIuzWHn94e3L62mbnh8BR9/EcVDVYvapcb4ZvDtqqXyj5bp9QiC+abkgx8bNVg4PXjeD0zbe31KbJSDqgIrzI2Cl4tjsCmeMygE1iK5yL2cVWAonq1WvCA4lmyM36aH1hBw8sLznVCA8NDcU8M1fT9czUsXJAeEsKg3/u2uAYjW6v2952Xjw3XU3P79u9yp2Tj4E3DZFnp+aVlsdOoQAWgBCqUP0ep+ywyHyQ3X8M2hFuWoNP4R7iZZOpB5GCIB3b/48y9sfpxKa0F5LFmuXFz9fUMlOVX3OBg8KL2WXTzz9rTrBBpMCox+ajgc18LwclA3A9QjfVtrOT1zOBrZ3JoeKl98mLzuQhgqPrrz2KigPhkxEyQUB+kjHrItMeEVm/CRHrXc9e/HiDfPcHlRgzrOjdoYTjOERu1Gn2U4Qneo5wlAQL26JgEYLCJBXR61OWMx0Yz0f6N4R6sWHQYHNsVOsG/deSzYD0mqDUALTVio6Q9I277k9+T95QHHmKM8mxulaPu4THCgJZbZMQgWlsFnaJ20kVSGRvIOgbXYd9JJ4ua4gGI23ZEPAxFQTb91y9nBnTzhgrC0p37Z+DfKVKzb5PbeoQRSsr2hy8Dy7JxwV/5Z5rfYhNiR7EjTghmIoHm+OlR7RIAUQMZzaigj9grwpocPmFtIGIuRd407KP7gDca53KleOzoBKzX5semdjd1c0s6zr+i4QyoIJpemsM+4w3qMnR79scdnRfY8qQLJs5LdVNC7FlpuMxIcPf7+unYZ825rjdZ6X3YGHsBGMZYf0oBIvkBE4yALmRidqBj5nKGRxCwLowDhDx5SP2HtIEF1jiouOZTRzq8vlV2hYUhbOp6Y3LiQFKSeJwRXrEjyjYMDP60JqbYEQW6qgmJuPSib4APLEsWLTBC5cvv0xn05ho8blWr1UkkHYn5Qx2UKzFn4lt1Mi6wYz1DWCLgNNz5HV5B1AEtX852x+Iw2QecX7RvAouu3ZubyaO05jChOZ5KHjvqYupaQhrO6uRbeu0HMx1w6MkEZ+p0y+Ww5vB9CsHVnBbocwI+8JHMoU5COgN/Rok0IP3OwkRJbKgsfuxEek+DkzTmAszBSihkdVRUknWS2DEjQ5npy/rfJLYCL7GF5+mN6itajorAWDjndUZNnHmJSrK9UASu/SoYtNByVzdf/zaBiYkSAh90pnK4TCu4HSmgVROLp1z9jghlQaEp08eHeqq6bnn7C6JokMYVK1RT+njH03N+/C3G+Aa4/RFvFN1PoDrOMsugjsyxh+zkiycTNh8GAVwoVviw87dLWUeVeeX3svUu+dYRj88ZjU1GSgECk6+gJAkk1NUZpwyZ8G070+YAoLxzWVpcQRwfup/0zlRVoxFQVNXujGoAkN0tBIdJOl8c8jcpYfsgPfwKwkyjdEJSbzSbSomKh4CiPwIk5Z6vOk9wJBtO0VHsX44QzBiUTR584nIKdmNlZthCz9gz+7V69219ds7qMQRHyzUMHno1dsj9q4zeNRxm05l2wGiGAusC4xzUeKrwmDRkvfUhNAxC58chEmZXKZrwkMabEM+/Q9Uk0m5ebjgutuIhtDJaZhiWtNFg2mGThYuZ76zpoftAYbaAheMgXFohBsGbavIPXuCw3HYx6xpk+Eej5cdSd5U0a+0hzm9QzU2mJW47MBHoTy9DVFzO4yi7O31J+m3+zUaTSqKdeP59ItVcBQNfiAVZbRhOJVR4PsUSFFGBLR+WQlROXDtTlMKb9xaEcpVKSiD2LILjl+zjIziDc6oIPFZaQn0Zs/Wia6Qf6WYtB2SJGmxW//A7wK2zJ4DwAA', 'data/primary_sources.json': 'H4sIAAAAAAACA61VbYvbRhD+nl+x3If7dLLeLNu6YsLRCzSQHOHuQgmlmNXuyJpG2tXti2W39L93VjpfmmKHUAK2mH2beeaZt99eMfYX/Rm7QHlxzS4M9Nqi0+ZwcTXte9OGg8a53l7H8RZd46uZ0F38h4WuN6gwltxxAcqBiRQ6wx1EqAceOwMQLyGtE1jIjOciKXI+X4IsZZlU8wJW5ZyXkGTJYsHjNNpCR+qOlslGhy4Y/14VF/Tw76v/+gQ939iDmZ/xaBiGGV2ZbfUuloPF/QG4MbBDGGJaRWEZTetoHhGovkWuBESdVoEpVNsoEBBlSZqFT3k0JPWgWs3lOWv2YB10cY0t2Fi7huib5CzJ5lGSxyPsTd9wC6KBbjOPmkNlUL6w7PRRnP2J/dFuB10FJlh9+Pj+/c39p83N3c27T49vNndvH+9vHt/M3N4dLyvuUCvebkjBxjY8KxbhZVGtltlClFwCrOqq5ong2aLIy3yVzud5KZO6kCkviqxa8tWqSvIsk2kJopTLfAXJ+VBsPUr4jlh8xY7UwneUYc/cpGnsLdE16gosHKkOweBKRt4GKdA3xmaTzHpZn8YkldmgkrA/g6k3emt4Z2chpenyBE57I2Ag4k18Xi1vnSZu/6/ih5GBW3Ac2xm3/f51T/kp18tlskzy5Kh2BwZrhNHq9IS9vWXEAmuRVpLxJ481GKZVe7hiSjvG2ZixU82y0RrryXDINGagHXPCNpRSZ30buP8M8KNdy4plnqxOuvazp6pU7gWnjKw3NfWdaMIvwQqDfUA+enoeugDJzY9GXizTIinzbyHnbet3yNsxNlujvZKjzogE+ko0IFyEqm49UIP5t0ennRE4DBsqiOKMM+F8csDxLfVkRKoGbA+R7yn+VFan0AZ9zDts0R2YoDQAsxvzISwc7N1PlEJf5Y/g3gavnDNY+fNwOyItwE3PwP0yRuyMu3B7nDPhRZxmcdig4lbEmwAbKerHsO+5smQw9IDnaosE73o/nTfcyIYHTkmktkChlf6s5zfPukOjIWeN0ZUOvdWyGo11rPLYhg7DuHC4C/SgYtTw8zGg1lcWnnx4+wJrqrae4opUWJOWgSaoNHwgxkIQTlM1PPWBqeQbfXJMnSfPQ5wCczNvRwbje7C+dbGluSWa1xR6B0JLWH98uE7LS6odQ/jJbetQ3PEO1nfTCLmkq8bdkvROr5M0ol+A8GX7F1zThMuftzvs4PHQw1rY3SUNj7XSJ0ntKgycEDPmQL3FeaOoJxEl2kg7sqF96EjTiKWjD78+MGG0tUTR55GeV7//A7bgaQCpCAAA', 'code/00_extract_public_data.py': 'H4sIAAAAAAACA6VYbVMbORL+7l+h5aqicbDlFyABk8mWLzgbqjYsBeRudxNuSh7Jto7xzETSYBsu//26pZGxibN1VecCe/TW6m49/XRr9vb2zopFnhVcEC15RspqnKkUntNCC0MWys7IjJuZNKckl/dSk6nMpeZWElONjVW2gsdibKS+51YVuWGNxlWVk4ku5sTOJCl5esenkuiisANSruysyElaCNnpdhO5tJqnNvHbJoJbzspV4wbW5U4cqDS6HJI/zy+JMiTl6UyKFlnMVCaddDPnWUbOiwUntSycp6XlKpeCqJygzA5r7O3tNZxOSTKpbKVlkhA1LwttCc/zwnrlG6FLT0uujQzttMjTSmuZW+ZXr2eiczI1XjftPAvP/zZF7vcsucVJYcNLaIZJer1HpTOYw7T8WkljQ++DKidgrJcDtkir5jIICu0Wwe+HIl8LK3kuuCHwV4pGA1yYoAtjQmfWlmbQ6SwWCyZLzqbFfcesjJXzDu5jOgW4Vbf9c7/bP2x3DzqJWenDpARbJRzAPDlsz1ZjrUQ7Vxax0LZFeGSgMHUbXg5/Gf1oR7EwarmSHFx6r+SiA602Ntu+3T5sp8W8zBTPU9meFyC70CqftvEw2/1ur49fJ36fXz6dn43+J9NEkVZzOMPasF6vUwFu29NKCWeCqEMBdwL/tSuDT2i72zjpslJMaOPs4io5vzgb/b65aamLqeZzwxRgUeTa715UOpULcIvu0MbH0ce/j65w0fWnjx+HV38kw4vhr3/cjJKL85ur4c2I2aUFm36/HL27GZ3hiSXXH4b9o1e45mh8/Lr/Kj3hQsrjyXjCuynvvzo6ODk47h0eHpyI7uRI9PjRUX/8mh8fj7sH/b7oncj0RLw+OJZd2mgIOSFmxkFghIhsDhoEPjOQXqOY1aNNN+CCH7HqZ7OilHlE9Zg2EVcTvxo/k0KTcVakdxhvCoyNMj4fCz4gE4AzF1EP3P0Sv5otMqa0OSAzVpWI3sit8/tB1FY6h6GZXAo1hSAARZzS4WAiiJEW2dAdonpoizkwVphySgzEcmXWFCakSbUaB5ICUpBaF9q0aj4rIZgBFYK8G17evPswNAypAoXjPuCbJw+cui4GzIBEML8TSke+YeIbXUEYyqUyNinuXDNY5eIZBG0HOLvyv96mGbhJahM/0k8IySGwrKUDQpHZLnxkXUkDEZLOOj3Wpd+CcDQSZD9SEAMLnDAKntQKrBNJZVPoDTzB8mIRBapgMNZkyhRwfHMOvv7mRNp5CfKcoQiAxFSTiVo6BzD/vE/RBVbxjHotrF49gcGh5pmp0HTgqduerorKxq+6Dku6hdvWAFt8D7AnUwNs/CHHmvmHFpkoyBUJbAR98N3cWvtjgOofAXTCFlpt47P2DqwoM55Kj4nGbv1wLDZW+0kgdGWlib0TLTobfhKjHgAzPubizcD0UuUylaUlI/cD2QmdAn2DLWWqPFP5XTRXBslqG3rfq+WgH9tVCY9LOP0kyfkcUuE+Ym0f9cXurXD0AnwchnSNHBchFtU9GAAHWVb2KSKvLfjUJ3/M3XViIHM5H0s46TspS1CVuMQ9XMcpEC4Bkp/iOcJpykxgKYEi07LXP+r7vP61Uhpidbxy4rXMJOQk8VQsINEz8sFlp/XOsBIyPBY4wiupoErI06wCbgiTOvirNssarAecVouiygQRFeSj1BnC81VQmwWjvaazKr8zED6fb58YtM7g7E9Vvoff4DeH8YcWefCw98lhF7M6oQjcUjiwJqm5jyaAG1nG9IulwDo51FPg0ph6V0GXwDNGALZqpQBqca+Ln+Z2XBnwYWolkoibyQDwn/3T9Q0kpeTdb2cjJr9G9HxIm7csLcpVtB1eakIyMCFIgujxezJegmniaaBRz8bT8FMgArkykvyDZ5UcITojiqCpTxYLLyzl8AB9lRfOtPZ/zUCYnpG1BMNKDeLLS28RNc0LqPVULuRyIzJqHVy9GZLw2s5e97BLmwzwGTV36fcpl8vS+4wD5iCyN7RgtnAH5IMCFHA7v+eZkVth9Uh1sTAQdug5XAgkQUvX48Rc/vP6/IzlVa6AMiMc9BITzxMwrSaMOvq++Qh1VWsC9Yc3ObIQsvWB435xDUzEldUIKi0ZUKdAYzV9Y/XbiL38ufmmA08AI1wO3JrxKXCtZNf/ga/zDQBB3khnOOJTU6T3rl3BdYYFeMa4KZdffgarlIijL2K/+Zl+2bsNW/C3e7ADIBSFouTrJ1jVB+TkDxwKVF7J9XAqswyMwWobGBAyPAdGQzWqcUTffP7X29t9VJ/CP9DZmrHplyXv0hYQPNCvVuUzHIeA2+EXgTJfrp0jnHO86rVTbhvPwsHp2PwpPtyFIaghieVjuMUYrKg5hAPPp1LQDeaG8woR9EidE+kAg3DfeeVz75Y9ALFk0WtEBx45cjkdbDvFz+3fNlvPTfUfWhZllTn6pAOnMkwGeb5wTpBGQv/B7bet6EEFd9l2URC/2ng4Qk0GEYwRQ9DsEm6DdCsWIGzPAPPvoX6GUwSpddk3h8iParTxMg6XMjbUU1fHX2JLR81TXjIuRMLr/oi223jdpC2QwqvMxpTVO+6YufhaursHbTnOVLltzWQG3PpbWScVYFruSAeKBgWr2q6iB1RC/YhXWzjB9C5soCFUYBcfiNgC/VCZ2NWR2MGwCZ18EeNTh7orKjQp9u2qK3eUle4qXK93z/TU/dTLt1ecZsU0hH6dfmI3u0PdzW7r7rZxwvVc5qQZZEMQFDC5Lsrr22UrJLbmX4pYwzACYzuUp5DSjUJPJygcL80Qnq74SpB+IuxhopqXJoIJLeS13Mb9JnoQoecZZ7RU1qeOoBeZAAUh+CB1bL6gIAupN99ehJjzV45QhwVj3NCG1Y/UX+roIJhNAyN7AS3qQk6ahFcCjBBJncvChDjeccOrI+tvZBiYYJ0CN99njCsbiiDsxbsyGUsgLRlqM6ysAMKFq49qDVxGMdKycDBekZ92KRIC+rlX1wkZsQ4XpqDmU2xjqQSJEYokZbMV8RUngRAxyC3fOXJnLekgAdHmXzjgUMBmgqEBiZXW8EKixssOch7y9ecovHNorQUwpELabEXr1wRPY4m79bsLPUxYX+lbdJ0769W3g41a2iCZbN1GUWPUwcVYsM1P3M5mMCeOn0t3TqOuKqcuXFzmwcXbRdrztB6tt/UloQsUSHR18eHciLPrl1ZJzeZ+Z/Ri67u6BHR05AR0mCAdPimwircG1v142Yq23ro4Xvxa8UzZlatiKtNx5HblbOr4KuFnvAFJfAUYf7oe9E5e0O3UNKGALcSG1MAZKr1A19WX4BewVlvIFvLXIu722vD3uPr21PtBxb1++8D3zuGWeYOcDha/AG6L82Ijv+6isnCiE4rW1nS/hiDI3ITg/01g/i2DhnD9izkNOJhwTQMAJQlmxSSBAtCnx8Z/AcuiLZPEFQAA', 'code/01_clean_data.py': 'H4sIAAAAAAACA9VZbW/buhX+nl9B9KKjfCurTtZiuAm0IUjSC2NJmiXu7YquIBiLdrTKlEJJsb2u++17DinK8kvS9gL7sKBIJOqch+R5fcg+e/bsyqhSmQfFSjkrMsUKkz8oLfVYhWysdJmbVE9ZqisIyawMmdQJqzW08uxBJWwu8YWVlZyqaG/v7OoYAjJbVortD14NWFoyo6Z1JqvcLJlOKwN5DBW5qaAtSzuWT5WO2LUq66wCeFqVNM1eoio1rtJc97N0lrZfjGLjXD8oi5DqRBUKv3SVLSN2mePbbJZrdnF6TrOnEMTik2jv2bNnexOTz5gQk7qqjRKCpTNaCCbTeSVppnLPD5lpIU0JK/yzzLVTLGR1l6W3XusKr15c17NiabdT+KECW8AA/hXJHvYysesXEzmGLQJ67h3uMfwYhdVo9oW/+/XlOT8cRIPBfsgv7Mt+NPgaTVUVlFWjE+EpLYJeVBeFMkEv1EUEf/XcFONMlmU6WQrrkcDk82aSMq/NWMUEg0ELyW/evrs+OROjD1dn4uTt6RkPOe+1wFatyOH6Da3ji6vz4eWv4urt8HJktXco/sROcg1vJ+nYun6SyanznVZwHWsD6HbJSlgsWyL01EQZG2+5Vs2KI2ciOY+bHcT8+j1nuXFLg39hOIyE/Oaaf7XCk1Sn5Z1KVhpvLjc1zq6ggeGQn97Q00mjm05oLhvkHuaw8Q9H0E0ybId3JNuPeG7HtzT9gMiNQDSaJe86nq/SiXsvKqmFy8gyAHTjxEWM52icF8vGykbd16nBVr/wm+HfxYez42sxPMWGrt7f2L/Hl8fnH0ate63v6O38/OxkNHx7KU6PR/Tht+Pzd/T33eVwxEOLvfrhp2cjiJOx7APpnQ8vhiPh1TbHHQy/+XAzOrvwIdJMvgq3lcmbbfRLBNiid2hkWir2m8xqdWYMsoVfpAhrBMYkVVlSHjL+gmKytFUk2FDHjwdGWrNF1DVCJMtqWSjKp16k7gNOZYr3Ipllwa6JL13J6ucaETpJM9UulvtAP9ZMLZDVVNaUpKJkFBwIIOQMm8kluyWtGQprElFa2Cii3bhSmY5lxoanLjsmspJZ1Hh7ESUmL0RSF9AAcom073gfG1xEHb9HrWACQamXO3fUXUCrwDow7IGky2Z/i4+8mJdpwj9hOTas1kyIXxt5P7pT1A9gM5gLzQH2oLbgbYImkiMHx1mdqJJNVKIMbI/amcwoS5wCStytM0q07kq7FDvppM6ymazGd4HhwV8Oh8f/SL786eu/8fuXr70nHPpOq0WBtoJZsBuWUudIEVaGTXIDwNW2E+wBuy6SqMoFvVTpTAWw+M4kCp16zJ8n/ee3/edLHiqasYxROBTqEO84jdCitNTyCT+907YByduM+l2WuVbISLXkGxFu8W5VNVdKB/xgsH/QH+zjH7IOL7/08f7H/SeMckqgLK8rWBex8OH6lffXyhxLJY0NAjtZUkU0cEThoUyaJ2ufYDA3GvC/0bSrgPFo97VE7hpB/XuxBfvzqxergUa0v+91bW+zOrJA4wjWm14oF2kZ7/sCSZyi8SGaNBY1hgtt3dryz8+LiApXNJNF0OnVDslSkC2gnQVxF/CuCrl7ImzQES/lbOo0t+rWoPGMI0loAY9K7jeSSOs0EZCPvQ77Q2MgdPZg0JXShOeXATG7+2haeSkssiFzIsvnigIDPGR+h+wN2nlCBx5ufNFJOPCsZQvN1pIfRLOLexRxZmvXD+O9xHh5b6rgoLcFbQXEbCoyIUuhAW9H/GeVpdMUaWvd1+mB1hsnvAd7tlWk7Yb24/Wo+dpZfIQUp0LRKUtS2JAHPs9vLX1PhCqkKJfm1Rq1WDg+IadTsHDgBQ2nCCfUxZQeL2OfibzhGKDJZxjK3IEATaLvaNNcpdO7CgeA6k6h4VkRPzlrMLwQEeqaqjqOBITpu3cj5jodlihTDWXbWsHKAIxCB3KIJGZTkyaWhjm6iFaSEYkntBvKcrQYGEhSXwUKK7F9Mpg9ATQ8r0Qbcf2YaDhGCzqHUA11y4z8fptu29gmyvLxR//sXfmp23ep5tEUzr/rdBjtkdYKh4ESR2AEmRwrPIcM3Oqvl2/fXzrBVV1F7qdZWi0t2nt0kmvx5vhkeD4cffgeNC/bAH5WyzL2LTtsSmXoinfoS3W4WX47XB9FbWryurhdBoT1orPA0G87JFqiZfwGzVz1umwxQqgFneCNg7U8DPkMzJYjo7rFoyPkikkrtkFEGVsrEx09VzZW8J48Q2SdGJfpv9Qu4DaPZnKxC1cuoOULoijvEMEQawu1n5mcBBbq7Bo0FeM+Dqzh1gy724b/S/v9Dtu1z2S5ema/2q3QRxcMu026Yc7u67et6QaetulP7GwxViqhm4r2euIQR+zPOp9rYkYS3Ej3Xatjd1QCNLNbZbd5jcIib1Fr2GtXBUDcP5fxeuj3rDOw1setbPdBEo/alCQ2NtDExH0EEoG+a6cOcx3bqLBNCAio7FoJcA38aVL7Hp3F7vl123MDgPj1/fl1S7Rgjy0V31gbFfv6iMp3dRhJSN5cjxScT/9Pce2bE0W2L5Ma7Ax98ndE986glT/Wve9D6fo3tbig6c+yiP3dVHRspmCiurqiN5zAjmQRySQRshkPeL9v8hxJChRJRJhHHFJmWsYQtSAkXPrrBMjGdK0V0GBEr70jWnFMjy/t4vkRjgkxPb3k9p7CDkSzz0lqAiAqKg4jU6tQgYdXIv9s35p2F69fbYBOo0UnYlw+BA7SyPlLbwyR5nMpvDvt1Uf5gL1QwNKdlCdkRMkJAet42Xrfd3CrYr3QLbUwbbxiRYve0f1OkCYmQGS2YI7kTg3HfB6ZlS7FGn4zRg0iolRZ2kOHgD6lDMtAdzImxyYv8TXL3NUqaNc8re4wU3tkJz6FA6zVak7JiCZq38RfFh3msp2kG8np0/NbTfMbObIz5NOkjJtrGr+yHYw49HcMkQPz2sQC6cx1gdhNh4Qb0U2sKEye1OMq+Aj40EiNMkqH3vBgcDDofQq1nKkuC2r2Sw6bGHwLtjxjJ6JfTVGGKcO7fB7zTE2QPqjOm2hPVWoC+tgmNxLdIrf29Yz+yA53o8jTX2G/7A4i1KuZNMv4C+WKSxGTz0t+mOHsv4ApETWioAF/XaK9UUNOl+MUbMtGB2U/2OmRCNPACxtEqaNf+CnhAgD7cFvD9YOPgLU6zWrXA3cVE6v1bwDY3BC0XzfrInL/EWEvsIRLscC6na6hH9PftMmTcdrcPfzgFBvW2d7qj8K2VKlrbj/ozc1rXZd0f+QvLNaku0fM5ipqt5tq3bapNfcLW3ocXPAfG7Fe8LvCx+rbSHeBRE+99miS0XUOFljPOuF9v4Xo6u0OUdlz18tNYlHbQckUTfZE9L86iPK5SWGCSi2qgEaipJ4VZdAI2dzDkeigqUaFoa0+LbcH5ikEFSAh4pgLQb1bCH7oevjefwE9xAlZcxsAAA==', 'code/02_analyze_data.py': 'H4sIAAAAAAACA71aW2/cOLJ+968gEmClTtRK2zPZHdujAXJxgHnYSZDk7J6FYQhsid2tWC1pRMp2J5vz289XRVIXu+14X9YYTNRSsapY9yryyZMn75datVcqn+fSSJErnbVFY4q6ErLKhRRraVQeiUYWrcrFuq27Zm6KrRJvi7dCaTxKU7fxwcEftSAU80xVRrVCrVYqM1rIVnkwrL8uzKbuDPC26qpQ13ilbppad4DK2lrra1lexgefN0rUzIUsByKi0MK0stINkFZGdJXMv3Ta3OErElUNGvmVak2hVX4gwYdYdWUpim1Tqi1WS95jvRIGtN7IspTXcjf/JCsTvKoqSKLuluVOtPUSFMYbffLkycGqrbciTVedAeNpSljrFhQr0GXE+sC/atfgVqtIfNF1ZRc20mzKYulXfcBPD15122YnwG7V+FcN9ED8a9Hkdr3OCgC5zxoEe2pgsSlrA+QHw2PcaRUGr9brYHYXLm529MToS3NwkKuVuJJlAVWqFNZQrKvQ/hM1G4l9ZJ2pV6vZyYHAX6v+7Mgukm9Bc62LPIgChsK/xFen8aCuIO2U8OEHNFKsCkWAWUmqa1Ne5Y0AJFdFVZAIeWmRqypTadeW+Cl1BoXKolJ5ajbQ+HoTfGc+ilXPylwr4zienbQS6hf/kGWnztq2bsPg74XWRbUejE1kddltK30igufatKGGfFQe7kOHP6aWJ/ZFXNbZuXvkbcfqz5AfZhdxVje7cOa5y2O1bcxuD0Nwm94VLC6xgqkTfcWmbQU6wsSSjvOuKYuMnCqcxbICrT3I31dKrIpWmzlrAVizugVK+OeHf35iD2f0vfQcHdk0bY2gkOSx11gstdk1KoSMZjH+h81fqxa0C8gzPA9M25GCD4OLnlV2QocphoftZfF/Kk9hpJO2vtancEFGUVQr8JvVlWnrUgt2gWUpq0sdTEjlsbW5gSNF4gFTFXbfpstdao0XLD7ATqt0XV6NA5Mz5UEF5+f/sfVeXICtSnpd3asxb589cdiC7pqm3M0rZa7r9lJ4vI6h/Dxg57pImjw2NXsaBcIQNte7XqQIvU6CrFZtv7R3qLtL9/jafhyQR9gDuz2Kfw+o49KEwP4ZaCHHbRO6ADK7XwZnfu89EgrYMAelWdEZDKqF8cpyp5ESnFIHdnpDgDv2VjATf4GF0A4dk/fT/2yXCOsztERsrVruI3LbwEbEwPDD1N5Yw+bUOCI5aHcN1VZNfL1Rrbpnb5Gnlpt4p2T77Ofnw4s/O4mQ1s4PI2CBM81c6EbuqkRuQz7nz5R0n2aqLHWIrKPKyEX+66LK6+sknP8SHR7NIuTIAimIE12yiLZFlTatSn52SQEJ8oOtFrKNrNZQGm9tVdxQop6PV88PxRLhp4SZcTDKSiWr3tVRVBC+N3XHyb5ANaEVsin2hcxMUZLEvgP8Fm8L5NdYfLSBTNSurJngZ3SrukTomncNY6DsryXKBsYfCSCHnPEafOhuiXIjM5wvaCv0AGBUEGILRsEgITyDgEswsUHmmBMaca2K9YZqn/W6VVRACRaqkCgkZOa1TNKOxfvOgH8IaYuAwvho+7B5UVccetZWNPQLYdHH7rnTauxF7g2TFTfODud9anZLUmhT3XAQvMck3/rVLJU5SBZ17kRqyxtnnU3C5M7vo4HQWxjSV7rFV8TAvK0bcgdefY04ljRxU1zVJuQFicfjknJyG98VcYnXY7S3kjIStoUOHbIZ5+lSbpe5FF9Pvsb81WYJYsH+nqZsZo0/co63IBfWcZCbkvMLZ0utWCNBCVc15AxObqY5EZC/rmcRuXAMUSKlIxJYL6E/ss2kqEy4nt1yix4EGiUolwiZrxMykKLqVA/UUF4vwVXGHGUeklZnv+4hcDFeWl/yTs8dlgsfsmLdbUN5U+jkcBavVej8fNavJVqXRKslNw9tlDhfXLh4cX548fxwtFv6M267zy8nr8GnSRLeKPnkw7tlbRerleXaXMz5X1p8MYGxLUtCoH5LCMu84SlPLcqcNQXUNQPwsgnEU/EObGGlK6OaUmZqWdtAhboIcUPdoCKAxCmeGBT9yriIQJX1ENDGWLM6IfP4bStvQhMR/7PnYyURM//3A85IcC0rapYkCxJeVvc/94uO7DdGWaaqPPwGyzyxGokCE5yYKLgMTi5Rahv8WpW1RPEL+Z2b9iKmkBfO5vw7q/3vWTTBPvkLqtSnKKbiWQWxKvVCsZ882w+ic8hSV3xhqXU4007aCViwd7aHsDkG7mDL6vuwfZ+kTlQ1b9HwvmuRPkKS6szm0nxXyW2RpX30Dzn6O08gl6SfviHocX1ClFU6zKnKTlj0E2p2Eefp5S6EmmakxHIX9oENYUciHyJhhF9jKDByaSj5Gvd6mEVFxYaaMiadvJMl7M4yvqxrg8peNqnbwrQKeJ0cHx9HWsGnjhZHfz3EX/Ts2eU1rF0Pqf+jchmy2aA6y5AZbfkqvJg5m1Go2TadsQ5Dz6UyajJPIHwf1QrNqzDUI3HjWtBsAUlAkxyRvkapVKCRAWQtVuratrAibyVsXXyUrXUBJ4SBlQxZfisvle0xqKjmJ6mLJXZAuAXKrson/T9oAlGSd1tPfo2taMEcYhfw42LbbSlno2Nv8+IrlR9epmjurgo7dJDc6IxY19MszqpOHi7JesGfgrSRyX1G522OoR5qQu8OeEhvkLnLr85eXJId/KNP6X2C8yRLVYV21ezXw8Uequ/Q9LpS63Bx20xOWa4SKtJqrk2X714gzeT1tvhqpeiaZVs231DBUlCwRRRpxiVKW62perZrY5i57EqT4m1Iljw7ZRsZ5/PlkNFej7JX0wDoNM89qAf/EvEeaZGquq2isiQE+jjb1EWmnASikTAi2DIlkOQzWubZrQSJQtbsGS+M5E3ZgBBNCxb/t0VQQJmafCtOVsG34nua0ga/ffkeMLcF8Uk07BTh+5S2reaYrn2yowaulBzinupp44u+C6rhGBAwPdydXY3Ae/qTFacWahx6E5rIfLmFq/HZq4H28v4HFg+AWbbHf3KwXqGsDZsmglBrNOy26mRFRMP3PN/3vXe5noqNL46BPR6YzeJW2VrU+p+tNC0CigzJJIcwOvtxxX0H2iuC6gsym3H7FtQD/bqIj1/eX83/XulutYJ7ULixgXEUlYgmNSYaTcg4JJ2KSrbUcLgwwLUF93PO85x/+dhtOZ3U9y4GeIgZ8bl49vpHLH6pEeUtK3NbSY74nQYllM8eO/pi6CCvV6hWewlp9WuyeKDPUWtVsceOKHSVGzqYnW8tfOTuic1ZnXb/xy9ttg7DHi6WSx3OXiCxxlTc+RoavQzqsVKFpC6Luu5u2cA3KsBGxuJKMfvGdj9RoFXa8xucaOU/HNxX6RTHL1M0vVM8zwd+B84WR7Ckx6DboLZ4BL7jv/0Y3xKxeR+Dc8j22bA7D7eH8hTwPjps+xwPU+86KHLdk1v8/W7RFUFJtkAytTYpzR9Tf6IRLsEFaqIoXyEzl0YOhdD7yg0ppHO64RyDcxayHmmYYgcsHbB0lGEHoVuEmaKKR109NUSw5t8WXD3lK/9AJH9b7LPtD7UuTHGlxKezCHCISUrTcccKD8iEvNzSuT38bRIyWj5XiE2sVyFv8rnd3gve6yzyn7PcfZ9PvltMG1muEg/ZNCsyfPoKXY2l/C1AnIZofcfRoD+AjS2sRdh3TILw+W/WCoaPz+3Hie4D2hiED/kGJ/CYMmx+jRcvgcLuHBU/Mf3d1e189rWkWr42/oCDgjQd0NiXp+Sv9PQiaJVGIaEDehVvL/OiDRX83KRopzlf8Pob8m5UnehZ9FVoV9Ip2QuedL3wYwxbX+oYQEGEXadbta3bna/OCdOfj8Pk5iXljnHNTuXjlsmq6mRp19hy7+F1PIJFp4E2All8WPZUnJXFulgWZWF2VC3z4VfeF2jlLkKmEUbdGBHw7gLBw1OxVHYMZtrObHa2IFaMq1TJTewf7z2GsPPQzs9Nb7CG6hi/blIs8VQtCW+GdorfUEu1XoeVV0cSBp9+/9/0X2evPqa/v6UzLZT0wSyqUhQvSdjPvFx/GDzUbfaFiao0Cos81Rt0JUDiXwAPpfdgRjXDMMOajTimET1pAxb3IvADzpQ/pV4h1oRs2TKyHhoKl7vJlunVf2PHleUwT7yUI9ex6hMdVyoMuv7IJZi5IcBeGTwVr65qxNFtcUODWIRxDmI0aNMb2BhjR7v3eQO7w3+wOWDRiKwItV11WdXX1dwOVLc1DL9ueZ7Lbbe1t+F1Itl6ZGwFT8Y1ZtO2ADwDToZFe2Wrd6hmtynBQgLjuaVX+CNk6Gp7gKds7yNM/HvAxTCokSYw/Ps/pwfNWeZpO3vVf0tPY1Oz/6BkbWl0SJKKUEBZyUSb+joJSrVCHXfqAPfatv2Wjm4l3GPf3vyT9T0G7lFiF/6RzLvbWutGP86BZtijs/yDxykmLyB2F3tJUsMPFjt93i+t3m/3+za5JSzrR+69KtaRvNFJUyJhd0s60dfhYXQU4QNtJAkPF/HL6GfnSQTWZh8kKk59HuirdbyqK0NxFS1XUNWVCmzU0Ym1fxpvp7ZAor25aAHhjoMqUJVyqUpq27RzHOrvvgVw1eAk+AiHvYY5kiV6p03rNkV2bnf4/s47Mh3s8Lto7HQn47NZG0lcsYadny8u4qVsNyEoMxMRHnpG6TihbpPg6U9/fbn82y8uRXA/HFXTzn28bHbiMFO6CqvnR4tFVERoq6uT6DudRSSBvewSRCQ/FvTxhCXS9k1ZbMPFhB/uCmbPDuOjn/aAE/thcOYSFzVYWzqG2U2iltfA6WipQX6lmx5C/JPE7A6P7Jk/ZIZ9nn14xZd0goHs4UVM5hL2FkZuE/W/Bs+Jgnoe3JZkxNwmgb9FRJ0kCgSNdnE4oX4UGe+CUTAfyKxbuetpfOJgpOlKAdaSFLh2JXK3KI3l+C+OAqejLzv35U0vWBvm9BTMSfM1HUBaLlHHU64Qy53YTbCW1EfmYW8EvwwGJm+IS8DBlmLdFJXS5+eBqRsYd0vjWToVI3JXBW8/nHp1bAgmLeUOgSGcndIrLa8U/r0VKkitQ5iATwePh26qdRDlTZEcvVzMTik6ZGWtVYh1fer1R+CjWxoF3arS3LSjb8lgb3VDZx0534Cwl7D8ZQVqPara3QqLXfUvc5KHTr4Fmey0LFN/pp+6KxcnwR/vP4uzT59///urz2coSG6HY5ThUlMbRyPMaTvlbzHlPM+YQ3c7Yn8lM1ucmhqVaNdm+65Y9HvcQ1FeyQIWVKreM1J7QAomjhaHR3P873jPuj7k2YLNNV06tbd/sPhOTBRuVq9tOLQdH1oMNCkdj0AbmpToU8EGrf19Gb5Ps/Obgz4qiLy4wp73cHWt+MB7dBpDkkS+pgsYuUARJEvhgGCQiEcys93iKelT7ypqxKF8D1NUEApfu7lDq6vc9IaPcGQBjzs5D76qthZW/8483O0xxUeYfQuX0fUiaYbdw6KOFkcv53QYIeBal4iHF3aiaa19v1HFdBEPqeu6LVAacWinN3HebRsd9kbJSbYyyZHPmS0dU/V1OGmhWofjTAzPYZihoHkA6EckuSPdyqIKXScqm8TfJYxfteuOrsR8oF9ofk5lE8s8T6V7HyKGUssGp7bz7iSI+5tddyDdOC/aqLJJgo937mW++fSPU1FvC0O+hNjncN6P0F0BtANwnZzTSRtE0WXuYlTdUK6l54uBwQnMvahdBzGgptJiXzkxwnz3473oh6laxMdw0FSP5/j4eK+gx0e2e5YtHLF2rROsZgXSeg219RMH+h3zhMEPfPiN1czoDGLSlPNYYlj5mHHAMLlObl/yHGMeEY/seSS12xG/trdAKdIdzxHsfjocYQV/9vBgaJt4Df+aDecmT8Wn/l4LyttLW6Y4L+CLMGJJb1yxt1TIpEpwqUUh3FCDVzdmzmcZZjMcq9t5HY+OdbLnODPKo9cJs9R/nN5m4m+TQ/gpbl+m35G9Gwe98Hc509vHD/dU7jzmYH4fj9qfNPgh5D7cB7ChNAWcStMkCdKUYkmaBic2phz8PyS5O1T6LQAA', 'code/03_verify_pipeline.py': 'H4sIAAAAAAACA61Y61PjOBL/nr9CxX6QPRjhBLg6wnqrcpDZzRUwFISdm6IolbAVR7WybCSZJEPxv1/Lj7x47NzeumYmdlv97v51e3Z2dm5iwZUVExGjeMrjPwya5BrFXJlcC5UGyFiWcpSJefVUMKF5gs7EGWIqQQk3IlUoZZYb0uncZExKxHQlTzCJJmJuS80NeuAyn8Ebjkw+sTN3AyzWBEjxJ67RDCRolD8Yrp+YFbkCcdel6qNiYae5QnGe8P3wgMJZMVnQQhRcCsVJsejs7Ox0RFbk2qL6R4oHUlohW2qphHXKOhOdZ+CBncKJ5iy6gsf2oCqzYoGYQapoSQV4CQT4UySdzm/D62HkODxKJ0JySn0C3uXyiXs+KcArZTsJn6AsT0rJPcUy7vc7CC4TbRpHTMFj6iyqJck8rvz2cIZ3HV/glO1XInYx+Il9vxKUbQuqddWinFDP+CeGyJwlXBM+By2NNZl/ojmkQ6GsE0vOVNS8wGGXVgSaMMuwf8IUkwsjzOpAj1a077w50gEJzBi0Kp/Tqnq8NthkDP+cMtP676LSVINnuJw0ZHc1RhUJOQPZnzV47N0t3yL0jG9G/6HfhoNrOjrDfdzFAb76elPdjwZhdTna4HJw/m08pKdfzobuWHgYAvVmcHF17ojn58PT8ejLJT0bjN37sLv378HlXreHgzVlCP8+OL9173tHoeO/vRyN4en21/1zeDobjkFKY0T94ESejy5GY9pyhuStt42gi0rQhsabbzfj4QUdf7ty3Kcrox2lded67Ohfbq9PN+mfL9vzo8tf6dWX0eW4lTS82la0eayKIAP2rxCRa/p5cDo6H42/1fTv+CX4IAm9vy0Jg6vr95OwSsB68MMPgt8Nw/eDf/s3B//d0P9/gb+H/mqbxjUTdV1loAUTalhWQLe7RnddR6XI4NV2S82jqp9J3dU1T32ItE3YwEmFTY4O3cy1HcgsN3b4WDLpzYkAVLoL7wlo1wDQNBNJ0CNHP8rZXXGWRcF1QLp/gdUpJd19VRDzqK3Xe9vwbe5qbAW4VA0+J3g7ptUJGudqIkVsqTBU5RbAWAkz5cl2RF+pagMMNDeRKmne81tFcv31ow598QPcGvHKxkyAdJVS0KhhQEIJaF6ZGWsOsXll5WPAmsyzNNXczWXvByrhg4hKrrxHP+h9cIQRtTSQZGzu+UHXR+gnOAkjEUxodgjj8snel2O49R7rxPnBM24zQXNNYcboBd5I58t2rNogunglZQG3rnpEQidMyDc6ZDMGJ3NSVU6wgXP3EQD9kmcm7HTd7GsmDATzdyZLPtQ6137/rWDPty2tlygaT5lKOeT0KQcr4xzGuhFuB6APgr0y+Cc0QFORTvckLEwSFiIISS6blQlWslxJ2F0srG8SVq29skBZaWCnyS2C6MFyxdrlrdZLlpKTaGP0PuNiZkSC+3fYoZ5DxAt8H+AUKL2we/jpMIBeFGrS/MCrWIIqrmnNZYEjhr8Zvn9ZlY3OZya6u18+uzWzDGpnhEJ3ntN24Aee09gN3c2FuwlD/76/DqYVp3U82vnhgU09sAl+jj4d+ptH3SUmqIwiEFZtrPbn2oe+C6BQJX91fhFVRu2GpPfJs3u1eH/X67WSxo2kX6JaFOLScBT6ryQ5lwkD5FPJKqplgJtmoUIlfI77NsBrWIf7i7WoxVxKE7X7GEl1XhbUigyAy73xNjLn9PlBEsxAcD6LvBD6MMiEooXm0eFKKOTN1SE0SlPIFNb2WOYGwMJJJczaoBcwm8uoy2E4v9n8Y1225xVtytEQ/uh1fQLy3IDZHmJ/qHymKJ9DoUPTtZDLWu7tkt8qzLtlDPEcyquYwnYJ98AMyFLGrnOADPhhSwP0NawIMGRUWbe78lfDefPC1ReGAKa+c3Cztiu9S/PBMYCoSm0fm7zUMf9QMn8SCVcxDEMtgWNqbWH6+/t8XuEEyXUK0pmJIbYMvm0SaqeQ7nTqtq2wewx52DvoVsvB/wBIy9J5YlI4/2n9zeYlwXbk1pRsp44/lgIEVNY3WVOAU06Q20IY0D8YmJ8ZNIi3tMTCsK8/V1wRZm5GQaVCAwekd+Tf4aUyi+/fr7yPxZHwXYGbnsEclfQhz62bTgWFmnKefec6h3ptUmEXr8H4ZqHslFv4bG5GSJ2MqsJRIwHxyYTHtkILrlwqkWt5hwdgC2oqy6yQWKs0guYEXEvyjICNrJRgokq943924euthtCTOoPuLm0gaANVxRo2bgNiGU1w+SxecCPkLXgCoE8d1omfu70a2xq83+iFCTaw1MmFE/bi/xhA/+NNgP6LMCl2SXjwye5CeIjKdQYbBCT8AJC6u7S+wem0hehtZA0qXFgD2FUlJAv49BbxWwi7Qarj6Af/io6P15D3nTW1Kt1aKTF8VXgktV64xM0/YQR4Jg/czjjsZuQo6MJG/mecLesDROT4iLpF4peoocWioaxwG+JHqfuvB0ph4FGaQRdQCllpv+4dwQOofHAryyKCCfFfZcwrZEwSAAA=', 'code/05_reproduce_original.py': 'H4sIAAAAAAACA41V3W/bNhB/91/B+YVSRjCxuzabDT14iYsGG+LAdhYMbUEwIm1zoUiZpOJ4w/73HfWR2EXT7kUi75t3v7vr9/vvrZPGqxzJJ5lXQVmDrNF7iiZao7U00vEgBTIqxAPiRqCd5GEjHbJVKKvgEXcS+b0BWlA57ff7PVWU1gXU/LS6p2BYE1SCHtwI+stbQ9CG++aaWxPkU6jPynbaJfjiYN2jUvTms9kya/XpDfwTxlZKS8ZS6qS3+lEmKS0hFBP8x8Hn3ux2mUWlU7xqX8icLJ0VVR4fiU/x4s/r5Yfp8uqCgezN7XKBx3CgxYNQLmktZUtXSSKflA/MPtS3tCfkCmnLRWJ4IUkMKh31kC9lnh2/mEYaWzlbNLFqm/Po/EBxXHypU0CIIFtrRf0kftJx/NLoVToaS8UauaRIx06GyhlU9IqsDgtbp9bKcM2UWUsfMGkS4W3lcsk6LqTgbMBWMuQbBqlmuZbcMMEDp+Uep72CNnlhl1fzzAeXwBXipXez+W+Lm8nFlNVF6TggH9y+tVc5nWle3AuOTrhbe3Jy8rCLhxHGPZEV9L5SWjBld7z2yHLItnSeQdGlTtJxeSxT7jwrZOBRFrgauFqZh5oe7JGJpCQi7XmyJfzZRhRrEdx48ImOUgANtCJihZRBHxO84rnSKijpMXBJgg/dYlJGUnQrxQFVR+oLljwvSh0N+Eh/Dv7IO9tW3EGseo/J9hti3JgK6kR4+nkkVhQemvvHmOzTZPUjpnDBKVFGyKfsPdceoNnVNisFtAUXtcJr1X8l+43dXu0HO75rq2Qc82KnnmNsxOjOKYg3tm+Cb+4WV5fkwppoz9VQZ8Wa/f7JXE3Oz4fnZ28G5Jefz9+9/Ym+GQ4+GfDih2Q7JHz43VL9z3AqEwuUNE0qVIR/IlbpqO2RduRQv+HDt++Sl6QeZpFKk1sB8ySlG/nUGoFQq6Lgbp/9g5ucae49Hr2UHr2fzafXCzjMpzfz2eXtxfJqdo2uZ0s0+3Uxnf8xvUR3k+V0ji4nywkm2LAWcHvm7A5saWkSwB1+JkNkrODQUACo0WFJ62S8wLV9vARcQb66EqcEgY+yswzwhWsLvQOP25reYO2AzGtyA2dWOvmo5O6A7Q+tAW1L95K7jkBNZdS2ghxGqdxWpu4qkKrP+2P+psoHZ5FZH455Ad4MC6jGxeDsoQDvYGMDExPMGhg+MM9rBjU2GBgOFOoEX0BlES0IZ8tm+uYbHscha9YWHrWF3aY/ZN1xCApdghqIHIgR3MXy7DgCvAkmZoWuna3K+32CvxogFMirv+OaihNL5YCpfztYH2wmiFPmD57GHXncX5FCRVWUPmmxWDe/CdkwhYnplPmOTM98uSA4fPZe+W+siCHEKhhASz1yKDxwO6V6TYzNV9eEeW1NAKfgykA//QcXirskfQgAAA==', 'source_original/01_fetch_and_clean_data.py': '...', 'source_original/02_did_equivalence_analysis.py': 'H4sIAAAAAAACA809bXbjRnL/eYperL0GLJIiqeF4RjH9LEsa7ySejx3JGe/SDB5EgBSWIIABQElcZd7LHXKSXCFHyUlSVf2BbnxQ1Hh2Ez+/EYGurq+urq6ubnT//neHmzw7vArjwyC+Yem2uE7io45lWZ3ByPVD3w0+bMIbLwrieeB6sRdt8zDvp1tme1GeMC8KvTzwmZczgA/mSZysgyIL5zqs05l85v86J5sCSSWbnJ3HN2GWxOsgLryInZcssD+wt5urCH68TeDfLTvzCg/Al2EcBFkYL9nbMA0ieOq8vQYp2BOsQb/G7OU6jQLCWYRJfNxhbNhnF4W3XAYZyHsWLhbwA5TSC+Ne+ZQz+/wGagHoxt8CvgtEMAf4MweRwH89fBf7Xuazy9uk997bshfhHeA8ByTzAjBcvn9x7rDbsLhmb99foKbTa88NHQbV2J82XlYE2eGfAy/jCBmzC2/jFg5bEJ5A4EHot+9eXrw6fHn+is2jcO0VAQP9FFkSAZlfvnLDwulLrk69KPJuPWLai4uvTuLYY/ZoMBo6muCnyXWSFb3LcB2wk8tLe9llQBhKs2AJ6H1WJJIrfxt7a5A9QIX0clLIPAHuwnkIb4CDFQtjNu2Nu+zg2azLVux3E9YDcpscWyeGilmvyALEK5GiUHFS9LZBIYukSGyZJZs0VwJdbGKQ5eQq8669tZTkEGWZr6gJz9jLGFTpzbGJe++DcHmN6M7zAlWVZMybZ0meS9JzLnqeBvMQZGD55qoX3KVgSmsShxpsHgVerDi6DWM/udVYClIvw1aQSMi6crYAYqfwo8g2xAwymRVgCe5cvj0G8Twf9HIYgQ6UjkPQa3blgemxFE2XG8kb4IpQQ484AYw39ID4krLkGNiEjgGavgWWlDGhGH4GlhApjKQG6OsRK66zALkHFFBODYDCMz9YhHHIhbEvvHUgsb3axNDcKfiJYttl49X6cDiAf0bwD7RAXhDnP2yw/+Rd4v2PP5/2hgP2HnnKr6E9fgrjVWmmqvMEWZZkOSh8kxdkml4B7AWgx01cbAlVicMWWA8J/TOHRWBckWpabnBY+6fQi5e9vwRg7cwePn/2FE3mFAQCF9P7MYiuvPl171UYRVg+GgzBpE7fDRlJH/RyD91GaTBZBj0RLZQz2cuSK/gBBTdeFpLogTQ2FHDUZ+/CZZKhW3u9AXX/cZsmwBW4UXaJkMCkfV465C54ttsg66LfyoLeJbggPy/9DLgX9iYOehehDzwgAnQuby4uHTI4DRE1OvXFsyAqPOiP9HcGOpXC0At3yCbs4LDHBv3RmK2Xhz+h539NyuYAIwUwGLOLM9Vs53e9twmI/goUtd5A2wcFqMa7igLh95j96gy8HjTis8GXLEe3CQLP0QpRRuEFATtiLs3hn5MQzK+UHto88qHJXvRQXmj3wTG7CoCx3hjq8l9P1K8j9Qv5HhDao74aBM6StQfu/UL2eOiiC7AuH9j5A3sVzK+9OMzXYJLYpVRvVvovIYbH7EfoOPEiDIC7kyWMTpsIOi5I9xPq7l1QhBkNN8y2zoLeiyArwij8GyG02I0y1NVkIP0ASzcR9M46sdExO4GmhR5V4Qx65iZbeNDc1DEALbu4hl6d3LKTKNrcYJcuSZ0FQUoSwwDpg54zHxybRAxVfwDLQbtSPRwcbppGYZA38HR0zM5vvDRBdm6wj3LH8wPQBgcJnP2c5uhN1syHkXmlnBIMEIW3gmG15AvhBSz0riC8UcDU02HEzKF1/nQE/nkN3ZalgbdSni4sHX4Dl0+OQQaqdAZ9EEYD7FhY/SxYYzOdJr1XyU3A3b0tnDzhhf4k+SuH3BT4C9OQhxCAqgjW5H03WQA9BmwLlcUOGIxM4x7885Sdvnz/ni3AWVw3cDc+Zq+8Yo5CXmyuyOuDVSW3MOqcgqcJfT6I0MDzIw6FyOIaBhxgUDInh0zSUo5RUhGBr1yCmUNvyQXa8G8oE1QGVczBKg5vvSy4Bq8UkE9DSrk+XCQbGAEwuJoHqN8c+hFGkJ1wDS1esCSXv/Kt+vnXPInlb3CA1wrkegOmL5+iZLlE9sVjvFlD6AkuJ07lqxSHgxzfpX5nkSVrloPSt0yiA+3nnU6RbXm31F+vEx/GgL6Xhlg9X7cBQOOuN5FXAi4I8o8nF+7F5cnlxas3Z+c/XYADucw2QSe4mwdpgQEk4DnHQeq4BfwFhNCBxhtxj/GolwnKIFwQSZbe4sObny4Utp9evj4/ebc/9Qq8IF+2QRolRRRedcqffWhy2zpZLi2nDgeRPf4i1UdFpyPaqn/l5eGcHMfSJto01k5k8cvXL950ubygV6+YWF/aXg5dch04OZt+aRM4hI7wOGNf2usgz70lPFi8FvQGH0bffDKVGC/IG/yRv7fBxvp54YNNOrOOQ1xBh55IU+ovg+InemdbL5Nbzz07dbVJQw6Sdt78fPn250v37OU7qJdgMxTXfT/MkClbPntXGAZd2667CKPAdR2n8/7Nu3+5eHtyeu6+e/PmUqsrYeXzX2HUsksqXWb1+1b9X0DZ6UBoBeFF4Ra3i8BV0Y4LITzXrr84BtPv4+TmRQYMci1tIXKECA+6OH++w+f8GMwrL/gbcNuFAgJerfQ2D32hZGwNo3ALRul+4HMPASN4McCuN/PhwOo4rPcdxKYQtRJkFnwg8gAxnVPsMcfIHwH6aPSrYJvbU2J5Bv6Q8wo/ppLHrmKoq5OdOSxcIK4c5wTsdRIHMyIIjgxo+Yu+nyVp7NnwnAfFRPLh9OdJurWdfhbAaxeC9ODORtAJ9iFQOuKI3eQKOYZhDus74iWyhK/hlWJv1o/h54dNYEsoZFdCSdY1KGRbvmbQ0QM24EQhLszdInF9GGTAvYO+KlohKOg1ArcJP+t7ebFNA3sRJR5M7SC62wS5ELajt7pP3IHZnMJwABEnBlq2IRLqCKB0a2itpCQUleriYdN0ZKdnLrZ+BoF2YI/GMl4lue5cH+bSPMpTr7HKpqwi2kCrx+vmK6hn6wJO2MYxgICvOO178dZG8AoGUo8r9A4qniLMrI8vbO8uzCcDpwavwFhvIio3wCip4KctH7uMWskGhvA18nWV2xyHA/91NKa1BtBs3WQftVToWiIbbJBRakpv1QnM4muAD2qLbOORGqtprWjW2qM1VzRqTqH4lg2D3nBkCnEFA8eqIxwmhB1+IAQ57rIBdyW/VN8Pj2e8xsrF3gcFAqSfX3tpMB3ORGmRYEJqIuEOlPM4YLJ1IL4bNvgC0Vd/KX7RsF+y7+VvWQqe6wYgQAEQNXjRsp/CCxsKuApwZoMIBOD3zNZxCYEd4aHB9wOs1EJPyf09oelUPH6rKzC8s+ZCYjl0YT2IwmvApXMU0kNrFly4v8HkO7dtrseu0KfTZT76ugn3dcq3LPUuIGlqtrt0pf1XpJmwZWk5vyxLzU95lZkq3GAhaaxWlEN8HLhUeUla3ixL74YCHZBEEJxgpMKBu7KWENxPFm4O+kSb02SASS1a+7Cr6xLsx3HY18y2+XCFzzogfydsUfSMeXLjzjOcyZeUAENpJsTo99JueKU84G2Rf8hkx8N5PP70Q29pC6TQKtjNnspG9BfAq+i+dc754OJSrC1m4sB8zu07hWamkGHUHyB/Q/jT43F5v+jP/UXZ8eklGsRiQgQl9SLPXZHZmJQ+A+aFtrTzr79mI6GWLM+rULxX6DCjEh/nx8ZqhxolPraXhL9jA9Gr+wPpaTLvVozffHRvHLaVBNKPGKzZHEsP24T8LT2jLRjcyro1ZrlBSF45WJ1VzFaucxcDNUByT5EeZwPbahrOeCYphKgMnBdIG8AMDae4gc3DFeejsJ4mHHmwP4aiCYFo+f2xpE1YuKHth0Q4S5jCx+xe9WyL68k61hXWLYsh+IQyoQTtfcFbGsqKalGqitJqEXQ161j2Yu09ZxKK+A+thPwAFNBf4z0NSFRCv4wy2VWpWD5oENTV6C1A0INWqHoKlKnfZjkZHS+mn1rplsZy61gb6DUbNwBzXweDiR+HAmcA3m0yFF3ho5hI3XqR7+Lsq3AXbhHkhY1TK+jnxzQbgXkGvNOmSjSRKTZpJIKtOzmPEdWmUut8CAj9O5rk8Jd9PrGYO+WUR6Hn8B/E/ALriTHb5XMX9G1Z5m3tqaIkjGw2nc+aEDaMh2gjHJ1CgsYzmwL28M4lsl1i2uH8kHrm1+FIuRvih0aySpQhcDsYILhqfrSgHgm1S0yH7EPpzxVe3ZsvyJvzqujE48kH/ONPFNeaqc2kc+OdUNYi7KKRs03MZ8t+6O+aI8MoPIdpvzZLpvRYZeb826bColwsunGjglKaD1VmyTiVlYtzEOWbEb4qAeuyeGLR5TlE6gJdZvF0p4tJRr1EBKvKcKdKRpxUSrS88cJCmEpbroHrssuzCxOhv65APqn6HTmXnMjUgtLlpHE6P9F+ExIxjIs4FhjTOkEpRhmfEAQ621oxulOQKnFzWguRyISTrYGDvcmImAB1+xNjeK4KdTUDjgyEfj7A8V2EKmm6sAf952MZnxBm0UHD5wMX8+8i+OlpCL6WgRABXYfLawl1UIUySI/rpL9poT1upj02aI9baI812j6t/XiUL8GlIcHQJk2hNflrWxBRoE4Z6hXIRwXyoBEyNXCKMEILCTWapsiOqK4TqnsiiURBNSMpEnC6HAcGtqlOVCPh6MoZjMliaF3sa7IfQ0mi2NQSvmxQkwF70AybmnjbNYUAu1RlIGjXVTsaUpbAomuL19DJyKjdlUt9+5hx4fL1wSr0s0ED8NoP3GcDJY4tSR1INA4ZtQ7spvPC1VQgUBxyF/A1Gw6gSSmQphf1GLoeLQq/CXGL9KBaSCiX1HGSi4GhdEx6ACjcJCXKMRQs3WZD0Ec+ij/MjGJwymUxDwRnbQGggFJvZk3BWoZro7bwifQOgpInDeEaB8SHSjlatCrHhy4b6+UUR/PSPKiU8cmAUZu6Q4VEWg4DCtYcHSoVpI9W0PJFhb5y0yYgvqlDjqsYx80YxzWM4yaMylW666WLJqFemJJTVxyN18uolF15swpO2vgloHEbyiLM1qSzqySJtGrsW74kX2MHu7Tkh9Mq31doaV6iylgDtOAM0bfxRf6mzpjo0gZT/F2FhNH3K6DibRfm2fqExs0xJtSsijqCmudQVzDmIhim0t4ol/ZGUdD1uHA1CyLXCE8/X8S6ctdhfIyZOijpjdVL706+fNapTo7UqotYcMB3XMAVRZ4rmrasygQdEelytOCDh+RGxS4wmUWVYeusXD4gDBKxFiLj8hz6aSu4cdf3mBpaOR8tjvNb6ZepNL1ffbT0NPEK839cZiNBTFlKxDvDAZeSlrrSZ/04KWLPdhz2B1YvZd8KpI4jUzyguzLLGEQaadDs5yT9nUDaSjoPPis9FKKFFm/EvgfDPnQLe9XlTSWzdHeVtUG3y6e2vBrOUh4749l/MgNy/KNnM1lyKwxa2bNQSCl02TJXtYkPb6NSgsrEp1KM1KTm7432tlbgqVZd8x1thkKFleOo6RnFMGzCVMdiY5SroMPEb//5U4qz2mrhCFetdrCzGk3/zSqVGR7Xi3LC5IidTruKSD29YSU4kSRgYDHCkub3VRU0lOqimsWGSEOxHCpYhqg2uOELMGqQsFEOp58nGcaOKLUNMuxe44Zu5aJeeO8jG2w2SOlEpV+WaZeFm/KsTmNirVsSkARxcH4ERfBiYh2Oaj6YXyvxi4SWexPMeTotiUEjmGxTmBycL5lvSlLY2nk1EdeShNOoahjmyY3smkbWTVHrlkI5M5Oymm5wAb7XeNIo8L6vcp+4NGPAIwffcx04WrVUT8S1rq7g+Fnycygp8mWWWlbEkd4cnbKXbdWijp4lB2MoaJemuzCDdWlOZqReglNf0KBRgh3Q0EZRFDRErcJcYZLWHw6M2igcDym8m6VrzEOUEqpBq1lHm5sIVe2GT3WRVLtoUaI2eaT+3jV1q0WROd/SDoMgRMS5nvaEV/EGt3rvDicx+sMJmwtjHC548AhZ5i5x27sRKKpXYm+6i3u6f0Pg6C8obhTM6uEjfQPgyl2LCDSF/0uWZv0wx6BkZmxy+cWNxdIxgGzWQkH2VLWF7g4MEvriglHQFILMxEpda5W2AEVf4RMeYytYNhCIJjLglx7YgJJPZOSjvMg/2Ch2lzB1WQaN408wh+xMBzKpifNyz/+rNeOalOhxZbBcHxf4p8MZ+AWs0yA4jMYKbiThdoZjomV4y9HOKPgb+PYU9wIsnXLpvtK8YgOX09e3To0Gg2cYW4u68Au30M4EDa8o3GXhiky7FmcRfsGBFmF5MHxv0a6XGJvoswJZ9C1SHJvhMmbOw3ijNvvLrbW0+F8zU9peUEZlMUGpGlMRUur7yDot+3owhYbcdIHdAzG76DL4Fz9weareQuBPMyqTadz3UICUy+quKJqE9Ib1LT41MellkUFkwuW0G/ujw/6dVUrA6yLvYJtCrY4z6xhot9zwCXOpG01LOI+1SJfFTNkyXxp0GjEhpYeRCX4eQqmk5j+m4s9juKIqgqlWLK3sVBsNfDqfkynF8a9/au+v1IdBWhEn3PyaKtSNgfcsnB6WTdVjJpUeU7iprMTXgo46qoq/rSWMiUuY3GFwALYiZimYjlzSQjSW8YrwyH98LDebKA6NyFgjJKMUHBBcf5Hv4yFgkshFbLR4vnMIJ8XkYHkzfjfRumel7NuJ1kln+mgniaHOaoOBXRlc2ig1g+0mCsakhOyD/y62DLQhWeFvHnB/CBzcpVz1UDqH2HKqkHYVMpiChcsY9ztR9C5mIxUsMJhQA6FPRMdcL8fkw4oKYRS8vYaZvS3Lqm3TLWsJZfTwZe/58+cNhGmaT9QJu3ovPbRIcEDggw52YbnL+6WWS1LwlBNowMNZeBCNtE/ZMUT90tJFzGfoW9Vq1fFnTMftnY3Li9WudBwWm/m4UjotIaW9LBsfc06fknJCmnefO+OESPddQpeytCWfiMGm3JNmnloCSrc2Mwkld+RrMVhw4z4yBwWYV7TARgLuykVVIY2clNFJViLkJ9c8LX9ZsmFnunfCLc9Uq8kXgWMxtu554AC9JXcIq6kcLcDx3NInvqBdXoBDyszZlRMVmEkBupT0Vkj7NSZnnpTCCfU+NukmDKLMeCHy5txbFZQzsisF14hcz8TtQkApqmYMBw9iINnmuVZvnjdLpQHR0y55KggNSdprkiDVqge1qnp+sLOzRVtyhKWuMKNXaTL+qqFltIKKxs28YKlRhV17MlVUxSlf1nKJSE184V+JnYTwj0ws6t6aZ0MKPg42+vHv6Nvbqh9HJ5Lz4xN+i0tVIMK1agx12ePc/N/JIV+5CXqsKNK8Jxdcc7WaGuWW2vZa3O1Wq8jN2DpkbWeTWEDlOHHCVk5Ea3tIbJNz/lk434SiF+gbs7sVdPW8YwXpwR5IdWXyxGe+a3Qpk8olqeYRRCBrGkNUkT6KaCzKIUKXhYaKI13N8/xRWiZX16CPef7JOiYX+CBKQ8MZOftPWzigyEBke/nv1gUEEUR0TYLlBI77K1ckRc1Us8yI4gk25qYS04bM3SVaJWOnyZ51mraWGH1ZLUZpPbOSxtbx4f4RQDRt4JxGvKfPn/Nxq8pek6w00LXVmLVL1L5l5AFlQKNVsvx62z/ZVc9M95dmUt2ZQ6lu7NoNzVsNAyR0c8O2QTc16R7rJkb3qnQr4zsVp1mktpYXoU6tGQX7DUpobnYOP2uUt73BtfDI/BKiEkJ0671TW6zw6VANdy3PNXDn18F8lWNg8MFcp6h9RZxvIr6m9FFFGR+4o/+gZzTweWqJYzJcOibDXQ14+GHzQm3vgrGGQVOPgcjYIKDcNYOJzTC2p6MuO8LFrlq0IvHiQR5gk0t3PBh48wy7u56keIg8jE2jrkCG51TcBK4/dwmRBDlk4wEFdrjxkO9UTHx3PdwvSvog46PWBR8jVqppEcKjJiE/RyhV21dR2bj0UBwlDGRqlbY1RCbxwCQ3U0esUINoY0VVRHPA4LrVA7G6YVV8Rw2h5nMkOhGhPR6XsWyomCtHliaMhovRWg+34VILPiRyo1lXOG3E2y75J6N8SAEtiJ8Yjsvj59i4hk+RuRDvg4t9mw4nK5kxzs7RuopVO0PnR+LttgpnHqjzQwBzpvmK2ebJOo5eQx6xIzEfcthDwQo3/pmZqPakR5xyJ5JmIa0ZEztunmwyqEhuToip7RUCH0PV6x/QmBTMybWYaO12KMbXNRMe/2hnkekryAq8xQ2YeWjdJXAHosrLmXyluadSdP4dL0jU5j9GrqyaG0cpkQ+pYO1o48CHI5d7PjXp1ccS1P5R2xAiVOPeuSYSo7SiOL4EWyFcDg1Hf5ehoaUNWaME/8fjQ0vzSvYgfFGHU7lX4lCq6jhB63JStrrDPKpN1uvNVPFvBsaqqzwyp/EPIlO6drVDrvbk0zS2vRA/yO4n4Ky79qP6rKmO2XTtWC/28JSxf5zR72euuBvjISNF7vGozn9wj/3/1zWfuOLEUpfOQ8OQZq6O36t2TWpvN054UJ1kZd2g3gMI+PGddReNSmcQFB7VfSU2jFV5q4vj5QK/LgICP16CXSQqEnACjxKgYkP8FFoyRQre9M8omiSobzLajb/yuU+Js9D8RRNS46uNJht/JOet3aQyV6YFcHlOnxGiyWP33CWd5qegeKAgEmcYp0jA1hBNp1FKuVeA9ojg7MHA7NGdfexWdbDmJx1KE+WzaF0HRgJCRWCUZVgGMR3bYJxHjUfVUZIhuAGk6ysMwivbIhkyJKM5eSAAJSvwuF55IgAPt9Oo6Gfzt2QF/U3q4ykRmkNaANv9hbcOoy0Yj5V7cd7Lgyxc6OE9AeXh32iDt77E490Feb8IiyiQpaPGUp6LRgJXSeRbVZjIuwqiJvyLcLnJgr6fhlAyGqjlIPHhxLLLsD6uBIGUoHCuu2EXUxNQjCgn9vAILLwPvYnl114WbPXlHw8X1xGH3Hvo3fW9u+s89WK7h8nmLuP/goUk2cT6/dnwxcn5GMYJ+h5zMuiPoZD4n1h4Yq1xWO1fYBxl9n//l3kIrWM5Gik8RtEelASGL0bPj74BAlgAcXcE3aEnHm9Dv7ieDGU2hRDcEALgVWfydPR09LSCo4pkWPKtTkd+E+dBgcddY27JcsrjoyKxTqHZJC1R6Bs+gR066vjKy2xtfwNVxsWAIX0Va6JQnyqUCSZ93N4CwolY9qxWFd8yNFdcrIuJlaDISq9nT87PnuELLyW7OMIuk62CjJ7GpnKOSuW8f3HOtKPSdQexS+KDXRLP80dIaQKTZHlPF23w5Jtn428eL9qOg9QrUuL6KXVk2zrpmweAH7IfxTG/4ijwLy7d+19/xZNSs/W9apyPH7/oQt9mq7U4RFvrBYj9jniyLa5pOrX9i9UXzMbT43P2Loj4ebxFwg/HP6XzW0UCgw5HAw4qKLcC5Ws+jrDSzMVBzl/8+uu1VwCz2CYfXSTX1EmJvSLE1C/ffoTnwD93KrTCNXbCo2foMI6eqcIlDHE2OhyjMx7rDkTBRsESl+2jZD6x6NNzFgWLAkAX6PWTeMLxoDemhn2GdUXlkfRjQ+nHRv9ARzb6zZ5s9HlcWdt58qZHQ3K/wVVpEcdjnVVL1bq7+ub06OT87HO7q9En+iu5xWN/aRtqkJD/ZniuH56MjwbPHy+leWOCcVVCVWDNdf3QV+d583PFdWtp8l5aa+3wX6NHOrBmG63ha/c4o8d4ldFvcCs8qkqF+va5VKS0vBCPqsJzi3WHjQee400h0iXTBwL/8x//iV8EOJbGA8Zs+MADx4kIG9kWJJObYZG5AkvdyNvCzEGsqUHYN3Tx7GLtVOP6KcYEpX/TDrOTfhovrRJ37t0EeDK0QthlV1fJnRvGEO/nuHEGQ9oSfh4lufoeQsSm1ci0jEnJyY1B03oo92ne+nWwjMJlSDcUaD77BxwawaB//ZUuPABWfv01XRPeL8iHG3Gooj0cPO0y8UdSP/nmxdHZQKP+tKR+QWez70d5AJQvzky6tVFjOHw2+qbi4EdCrXyigFs58nRq0ZPFP2/MU1wFKedAM3UgkwKn5MQuaH5ehgRXx2k8WOXaqELnYLTVEZ9QwcRVfrFJ/Rs/7+KyOc70+Fh9vIqn/Pl3YPfmKX+EwDG/vSZRpwBdLpPwM5O4HGaJOCiJ82sWQaugrdpTOuYDAWddNt0CDzOtkc6fDE5eGI006msbEiUS/GYdeltiNQzomp9/WqlbBHeFTTziqDSASSWiWVj3V8cH/SeLj2x6D9zB76PFxy67R0j+MAO8N97E4sf7Wy2xEo/YuH/lqqwV8MYQbaKhea6Fa9zri7hvRHHf0+aYll+dgleb7A5Bv6BvXJ4PvmQUgdESnE9diu7/wf1HrRGqcNI7rlLRu+iLJMNbR97isfgn/NacC3FXzgWes4qOmo+RF7QQl1tNES0dnWzdWY+MbekgJZaR/9w9Cu3286O9/PyI7wnRrwdbkPTt/n70aH/PD+/vh/EisRfWBaDyjeuksF2pTXlGI8c44F6NLB+p/F5R/mjJ8/TRd9iip+s0LApX6GowdSPY2LhT7ERcawYFesOL23n6eGg/Z/3Dw1oM8fqB9DZXKTm67kHm06Jtf57fSAP5ZGzym1pC1dG200tMAVgbjKGcX837ZV4IGngRRsHrpHiBYw9dJwHt8CrM6b4kxSjj91Tccxwf++wdBJKDobsIivk1/yoZr8Vy8Y4QvDtuEWZgJ5Zj7OJJ/T50Y/yA+UYyo77qqhR7opjnVWEI4F96L2lMUgJAa+JWRP0GKuNGLYyMmlKiYgGz3JqzBgz4mmcytfRxCwUtvn2Ahp5k3UFlXMbHnybBmC/zNKDWY28TN5uKO7NO5fLTbF9yclnpkfTEzV2fTm9OCHYT1Bvn0SI2rZ3tIaRJ85FiNtFsEnQ0+K1WMmqXSF3PdIhXRuFkpXZN2yeRFEdONdJ8BYG+nNHQ8Gzreu2yho7m3f2mdnub2ZLgd4xnZJyHqeKVO4HvjvelyHftiLMOMXtRbkLSPsiiIKerTgjUDgJk88gVJ1Trvu9Y38dTXSHia8VyIaiOdKKh32sNl/Pg6DTLmcOEc18uGGlSyq9oMnVPWGUMNi+1sKaYepmxe8L48VgeBnuPBL/Ch69mInq1L85VQR7Aa3zbZal6qW3mxVIIbB1olQo9DBNPX2Jb8EpyviLJdJn2HkNkWTCr40pdDE5svrte46LcYot8jBHpq7NzFy++k1DaSXVCEn1fk/bhzrzL+N7bInfnotmbT5Ur9wzsWhA0DpWbtO5UfWhFnzOXaMwlf0fmGnrbPuzhNmW+WVn7imDXgSoyHnmAy/IL5EnLWSmdejuKNF+5IATxDl04xc8wySf3zUdRNS72NB5O1bC2s+OwKvPFjjr6EVaVN821jJOttCcNWqxI9mFmvjQtBK2qXTO7tKMniVsPNGtMKT9wsllFUw/V3KWvHXVbtabX0b5d1Nb/44m1EiYnNKrbPy8uv3WECPvh2Ybei/VLfbU5i2nX/SKh2F0j0GX883S6iK5hxtfRRoHzfz1/fckuLn8++zN7++68d/nu/PXZxYxpQYe4C/QF+tDSJYLDrR47JZy/5q4Xlp3uqEbKlkPGPxn1jHCgkYPk0zhI9uVAv89YuweVmNBdGwyKjZ/ntPOyu7bBUt/waPo+CuFRd3314aijPfEyA0rp4fibURyU0S0HepgE89dsqg5Rpj0yDV54Nj3+ZqZv6Y5piwf8wPAI84cak/tt+e2HRbDObW2O7OF9cimOGrhZxObInWqxHhhBe22u+DXBx+xesKWN7boSZKgk0Dgd/XwQg3CbLPvsq2lAqzNsVe8alXeQnkoMnTa+DZyikffem2PuxukaBKTBbPALtqbP/8hezDu28TgZzYr044QrAYk2DB1rsdWuGroL1uok5jdu4pMo+gqNc423wmidrI0EXpPM71ipOFRq/CRDz4vR0DzJ/NwyNs1hGmtc632oDV295j41fsgcXtC6xzigNaNAJqXpIwYxFtDd6KAlvEqxgh6Q3ELMFMTzBPc7TKxNseg9sxzMxi7KrobwfX+zTm292cEu+CgCChiVfoQt5Feo090j1T4S7FrX2icXqn+gkc13aZO4Lt2HTweYGdDmDaO1GuAaq1eOAkkHj3CpvvfVVjbFHN28Sx/SjbBaFxmQZ/Jfhzndc9pwqam6AFVpv/AgxuAnYvI2GIzI6nUFeSKR2k+3qEWA0BtCL9X0R02bRD5dDM2muu5MzcxM0Xz0HPMsTGsK5di6Osu1KyOrAitltClW0Go4QcvQscLT1RhsiIMsPRmtaYlJLeFVzTCgFAHlVDvAskuSuC4Njq5Lvd+1jsXtoZj8/l+y9cvl+oMAAA==', 'source_original/iowa_data_centers_panel.csv': 'H4sIAAAAAAACA8Vc23LbupJ9n69gnYdTSRUpESB4mzdbzsWn4tiOs7cfXbREWxxTpDYvcfz302iAIChBFyqeml3ZMi+gCC42Vq9uNPSUzLM8a94esoX91G0XySq1y3VaJU1ZPczL1Top3ux1Vf5POm9gf5Fiizm0tedlW8CfPGnsvCzsZTuP+Adx7bpJmrbuv7V5W6f2un3Ms3qZLh7W5WtaPaxe4dJi8ZDMq7SGLyvzrHh+eE0aOFeXbTVP7eYBerLKGpvfuqibqp3zbdG/rCyS3M7SRfKQFb/SulmlRfOwauuFLS5/aKtcfcXDW5pU/d4/bVLBjfQv7lr0B/pG2h27Zvoh2fC/Ls8eLmYPrkvsq7RJrLO8KeG8dZHAzgx6l1aWY523Wb6AR7VkqxsA8KmsVrV9I2C2ZtB+3eaNdU5s+R32TZm/2IxMAj+gthN7ExZEvu2GbuQSt99wY/taQ+crAF/V8yRPbcLciWuTmH/+p6wWSWGd/dNmT9CnD11Hr9oim2frJLfu0zyvP9rUJZ7jMofEcjNwSMQ34RBxCLM9l3/fsmnW9X9Pp4v5KllP2nrKDSCtp1n5mkzxSvy4pfgHLuj3mNhjt0yhR49Cj069w/h5BvzCSOIXnAAfHQuf77i+Q3y5GTlgG7AZIqiuzQbwcbjqJHtqm7ZKJzD4ptSlwdQNppThSSfJnAU8nDNHOBwYl88weGrEWNxDoOoLVP1bBD4Ue+EtVRh7R2HMrH9b/mGUmeMbcI4lzB4dgzONEGd/LM6AaOxQfHgA2XMIbsYAMoc8eEecQ4ElIhsJZKNbfKux2Is1W2ZH4RwAzuFhnAMn3MY5IB0fkBNwDsbiHOHADwS4YMQuNzvq4tHIjt4R50ggywbIUrwb7lFXw9m3/3UM0JFtxYA1cf91GO2ID9BtuD1fwO1HY+BmotNkJNwUUKWcZ2GTolkj3B4aO7w+d0i/0HeBZD1JwB02SQc6mRI65QecpCjAbc/T2inSVyf9Df69ht46Tekkoh/OPFmtW3F+mVSLZdI4WQGbxcJ5rpJFi9SOncOPW9E78U4ojgQqOJ56t556Q8FxPpFan7o+HeEeqeEFde/HP8E9xv7I98P4S6Gu3Ay4yXAbd1zKX5X/fsOBIspMwCucJuzhcAjEXnBLFNhAJtm8KuvyqbE62K64XEuywvpwcXdluT7c+GPfzN5sZt+DpLIu0hqOZAUotA5h348ZQhzRmFMON0I3Uhvw2DsgDji4CPN5nhaLdGEJpKdned7+ypJ8etdWoBlT68P9xdX9/XR2eX/PUeZuM3LQ3FwkHeHrCAccyN33dZRfX18n/UBYvIFezeY1wp0WUzDqerrqHtpJ5vCSAWKgL+KgEOWm/gpP7ixge4VP7uC7ARscvBwxSqbYPfxAj+AKpnIFUxEieItonjcyvBoBQFlYN8ukTi0iX1E8Ja7pFfXNye63RInQOSGj/pi3hGCG/vi3BJqQvxopD91QjAUScK6Ct0SIu/ma5nmaFGmRVs9viww0dzZv6klZPU81nPGF4AuAl4Oy3lm2j3gPoRuHKhK9shgQJNDYJz6MOhWoE+BKth91ugd1V3jj0ItHgx6dMDS4Ew6kg3A5F6G+5L4i5KMkdv9/h4aIGzbdNhk4DPWSwF6+Zc/L5tPiOdXA5cOho+DPMqL82LfUr7m7MgRMTOr9MD7oD2ZlXs5xxw4mYLJ0fKxEpdl7XPcLsc8kTzF/RKhE9oRKikwIMSMGpswf5Tn7zTGhGlrW1JJnbH7Guihfiwb+t03GHAUCuuAIBtGg4/30Jr7dmew9D+lhcIHd8tt3pB4qhNxQcgRSvVDsVAcLLHQFUXkK/zIIzrOmbdJWUAXYJcTj8+W0WaZOCrF7CSbsZKt1Mm+c8skxUQmSdogfiKwrAiRXiHpXiHogdCUvQWdc3lg3SbN8Td5q66/qEcQQjFtdxegtbK3FjPR7vVHSWMiUMDBBy3ZBy0Hh3bsBIuSktbA2QFa36hmCSvPDTQkut0+u2H0I4I8xSrQ/KsxQDFzYYwMT7cHy7J9VgoQpRsrPdL4suNG9AUDVizg7bLM9akMiRi0zR4+gCosiXQwUnBDDzPdGajifWx9AYfO3fgwcFEU4D67hP/XYzOBivoCmhdt/SurG4FH0szs9CpGGsmMM/gXeooIxLXNW3Ej0sIPxwUhYMN6zSB8SyXjD57y2GXD8sRPHgJ37gEHQQDVUfXuWpXlpXWTPWQOvTWM5PYSYocvZaHtZPFWJwAUUd3fy7spIdlTkK/zIrGlNBueLEJqDu5/rwMT4aAMANzIRhyyMDSwssDmFL6yz4iUt3gbkI058wtcwaGXLP91zhhCI4XOGkeE5PZM9afwDwx3eGGeAWV4C/TjfynItHjqHQAWuwywujDfRxZv7u26IdWFslwWzSTQGCdQMIq0lRp5CJZTPq1nGXdk2y84mDNBwnYdtjJYghTNPrOyzhCEsYAjRYViGI4zTMg8ebWlKHRZ1Wv1KK0yvV284ltZw/fytByWQAd/APCL7Pmlf0oFXgjexXoMbElh08cVHG49qMV/1nBTycvsiyfOkRhb2hJqNSXTQS2kDIxJKzzU5KsU7wDd4u95RhRjldbmOQOY6MNPM44cjU81aeq7XmHQQNfeuKt7N2UjIGBEwOmXex938zVtu8XcPoi8FaBQpewrVBuC5K0GHqoScEhTIdFwgEkidygIWB+8PoJLAf7+0RJeJw3sJgImAW9G5ghsa/3Vn3WXPBfe/wlan+rhFJFUTrTEn7W3rJEIjRD4bo6G4UQQjTTOQ+XqxqWXx4y2tutMwAxGT0mGEOsjPq1CI6sJ+w7Q4ZIgB0AqQfJH+NgZDYBL7jDKQORw2StpjbuCEKDVUoSnfDOVmJPEbaNDH5HeT5miHmtEhjNrwJkPcBrn4HkWqDe+/QQ5epZwSp9bfWY2BPKjRzkv0o9twUl2rI+gJ64tZZBzWBsHAcF5GCFRla0qi9h3sRWrnPwNMM3K9Sf3jBITJQ8AXzK5vfmpQdLKJH71In3iAxXkQetxkab3R2gwDFewWB+RYGKSI9JDaxuHgSwpD9XCykKImqX5dr7J2ZTlATFWVFnxwCW8pskLU5ADkNUYBL76GQ8RimYmLTYIi2JmR9vsZmrHjLeD+E10pbAr/2eWKuCv13zkVZ6S2jYkbBb6/D/yrBG5f9ujvBt3odeXlAvVQRE6czkbA7p6YpOaelolpR74p8xligoAnqd9xKoBoXlb5XG8gcdRUABDBNtw/QPqmBaBogLc/txdcEUVEHjFiaxr1QiRjSnwstJisULMsOJ8u59IOjn9PapCNaZOeC0KZiZgqdPC5v5fVa5IrDyByFnoLW7bQB3ogYkhRmXEUKqiUI7ovpdP1RAsn0R24SIPU/fPIASj8S1k+8zAhXSSV9SNZZwuuM86zZ3lIjzVFWwVG30ie0L8EBEmB2MQuJ0GIP4mcN6VdQgc24Nz+PIaImhmmQJXHAF2SQwfKSuRyrA+D3ncuA2claCQ3RaLHD4NjbIdpk2zKdrTQk8b27c+73aANgy5sq+FY44G+9Q3ZhRwVyHn+schtKDaPGCN253MFAlzFpkb0ROBO+sA9su1wZLB6KIL33BEw0oMwUjOMURwLGBkxwmgI6xE7tLyTsMOgwZPh/UmjlOqjFKh29u36ZogT8YlveV61sM5+pUWbWnf3hlaXZ9a3bzN54scuO2NY2gBxVUxPMDQi6nY2wDqIUCCKdvgmhgV25I+Qt8P5b92maEdpN0le8pCpBX1jnVVFmXfJoI3coWQv/bLvLddFwH4zm+9LqGAwofsDWwpNSO0keh9ngCQSGZiAovopBLp5W2UgfZHLFCKEa/6NrOufiYdAYaWh5dmfQPr/SqyvWfKaNMtkmFzEc5tNbLWhYGECliAywRLumy/iwZuCYGopjpfM3t1qqptRN+mszbNJ1UV8OdHhHRWZ755JVnV0KofhMftydc0F6mrFwxN8hHo43AazQtuN5bGd4zAMhasMAj8wDsQ9wfl4B8mjgy42J31sTlWa4/jYXHyHiM1V5YM3mDNSsTlwzRmYL5f6eqf6x1FzvV2772nzWlYvtbruxw78AkljIR2LHp+1HImeyx2kgMztJ9q6rBs5yv4QGzEtTkTqFPZ21I94gf05yYA1fsusrTUDT/TCxWu9hu4CrUj7E6wGjwNDqrG+JdVzCpycLKxZWzflCmxz8AUAp/ziHklKBM1F+13mpp5FfaBG83m6qMr5S4di1/ttLxn67+AkQyVleZlbllvnefv0VENoKWUEQUXdLD9uEP7sHMbj4Br7pmwazjtNssp49mZCCdcQ/iRiFEiZgjYAsu82QKbtiuRDgrNw/PGusprHztmWA+iTj7NzKf3vubl3c+WBKMJ2Q1naIWqh/O3q1j8O5/dMjffVTQrw6BDgFPMj1geccOEV//U29KLNQfwJMiPgD1CY8N8fRNB4EoJHOPktcE7sssC09zVeN0Hx3lkV46S7Z5509+IN8DYKLNW7wNB1G348fBh+X8APTnCE+QsJLSpgTwM+wqA/kvXHsoSB1ztFOBLi42aGZEExHSSL2aAySbknBjrtrWrraxCN10ANG9atTg5bHSYQt0OQHERQ801xN714Gnyxqhzmm5GCj6DnOkpwd4Vcwzp3z1yNDRGWmGfdMMkPXcrgW5knH4cTs/qpw4YYSB4mZh7e4Zhi//AM7TZ8VCzh4G6GvePEmYyKB94L3MrPZYoi6Gv7ONCRAEX6mEGc9zl7hN2uyW3LHXqG2frztGnSYlFWT/bdHCATxbp8nsedsEhVv6llAsRl+yR5KO2N/1uvN01OJqcuz5wzkBIgbYtODhFpai4uDyB6pe5xxUZj5BDzbDHZ/7kqQeV0GhGgg3B4DX5eU5TGlvbd5+9aa2KrbQ1EuAsH0Q/pGBCFojwBRL9X5H6vyIMue0+PAVGuBiLbq4FUgr4HkelTrQoLnEaU5meclFVImTCjXiBBC4OxqNFTUGO9Eme9EvelFR6VLe4iQTIstxxg2KMmV0b1gPV0Y6pN2ljNMLzWBKFY2ePC38gI4A6W82i0Fz4+456u+zwDQzfKJ8iiLQGzWs2fxeJK5LenNKmzR4zKnLppF2+ODP26VOkgq8ACNJZ0DvRofS3rhhOsUiRXMAqzOXSitK74coNdTeHE3deZ3pzY/Y79n5JPTTA6oZ6ImknE5x/BDbgiWMENN9hZDYBpK6YHLBdwdpU1aT8lqd1ci/v8Pu6LFdEZqlcPER0dEJ03ILrep4ZHgokVARYdgSc148kknP4YON3NXM5RYPq9q9AWSaqVA9EIwqN7CK9Xd9GRYJ63xQsc80ag6e1Ak/gST280oMFoQOOeC2PFhVwwh8dnJYyKj+6Qy7GNuoQnusA9lO1vLkre9HSOxEs1s7VmvEi637Xvy3Lx2FZvHDgWB9z5BhPGbyI1nhLPR3pfTTIrofzv7QSP1iMtOaaJmKhPjsXHu2KZFqPHpMV816xntK6pvNhOOaNBuQtVlyCoXuCPxZT9MaZBT51BT52hpE7yXsVUijp90kW6d0nOl0TsnsvEBrb41PHycPYymNDIhJe/M+Bw3T1oyQGskmCirsUdVd8zCB982uXoQRjVe1L4/LSNH3fwqvAJXbmMOHBNyya9fRbh6znnu6RYgB18qUDPaEU92CGN7j2ZRxnq2+2M85/o235QeVpdHPbkDPqKrHSfDOuo+0o60S6rUNldzAZwkUksCyACSoxwGWYTUROAn49HgSXnyDwx13+yZTCZK/+c8FH5V8MZJMNawtkbEDLEvi8DHIyt7dnnv3a1ggA1mb9YX5PXF25OPk5aA9Ew06T1zkQHkQTTeTvnIv1VFnoWWYGk97AzLNq7PW3VS7cUC1XesUtdyICfN364QsHq25ff/768u7z+DkiKbEJZWp84OOsqq1M9+lSnRQSvdrnz63Y2UGQxVtcB/zAajMGxc35H4Kj60YHI+uiT9aPT76LP0+MozxxH+YF9WTyVUmpdgJn1u/qZyzP7vAQ9JT85Z8nfmIhUlY0qBGa7FVU/Ub1de4hfvZV9DxQibqAQ4Vl4cvQcoxvgx634EpFFFw6rz7D3Diu0b6CnKT60D+68Se2Lyy8/z/XDW2AwWe4W0MNoVG/rpoSR5qO2PBIH2md0RO5VbHbJnaNkelcvvqEjh9XjCofI/lKlCZ+8ekkHc4MDF25sYn/5NsODYs1Zv3WRzV+yAuvY+PKyEBOJHlXu3FMb8Ex7aIoPr4vkpWyS6TmMMZ7tXHLO11BUN1Vjq1/z2BePY3obwYxGrG+gxxRAw/iQHv9LlfG+5XtEAT9Y8VVTZ3bX2r4pX9N6maVi6RDWh9BJSFlHQaHa2M9FzLAkT3VJmNlFum46eUg3Ul4acqPkIRnmu8y/hRPskNwX7WP7T5seobdlS3vzL/pBV6TBggDrxDfjPnKIwXVtnawekcyvq0X5K5urahHrQ9dXTWH5PXHHCk4psek7BdSKsQKCK4r4KOgW350tkqzCGYBmKLE2GpovtPVPVUPNJgEzLaLxTAJcVEaK+mk5UDfLSkQfB5WCLt2u1zpcIKhJLaBfvgyYTwldLhITAPK0avYlyWEUik+4RJiNmOhkJOroSAUe3Gh2ystIPaw20Pg3b+hJH+fbT5WTEJJjxR7IxmbIxl2y01ikJ9vbm395dR6mR9wJZb4h10l2BVjyRyW2k51TWVvcvWktWdx1o0fDjccvrBjiweybtISWXWUZ1/V6FYY8Cyoszcs1/6E6YOSyXXcnBs10/mC4eMydRCw2sMfOmS6xbMxEGMoq5J01s/AdeDj7yJ90M+Pgg5Jdgni1uOOoekWsm7/eQnKqpoV3SmFGOiVsFMI7yjeR6k5QweInfChDwzh1vU0QqPIbaRd8FFDrEi7aO1C0alZvVzWrrEBk0ZhqVrGm5NRqVt8B9bE5TXBanU7Af6FooYbkNQgoHBMZL39fy4k7AKhJhfzVG4OiagBOW14k28nd3mVQl6HL4JlXoE7m8l+B6TZ2U0rkGihU3mrAobxYVdY6/R/+ToUZvUg5j6/thqjb9C68gY0fmIrjqpdEMWazCNZxENBubr/BnWnS1gbnQgfrtabWhuz9gF0ZQET+0M3EHRVaP9JnlEh9RkbzOV3tm2p9WSxaXm4D7QXNdie0fI5BrDFXTLiFbmyUahKXTa6lu7jWLM40qh0u3li2K/7DZwUYR1OVOZoNwrQfpRD8X1LVyyTP+S+56A+PsYcRqcEV3YAaHASI9H21g7GmjASiyDdEAsRgQDiNwQ0BuW8jESpxGvapB6urgiYn0Y6YkvxfUhq74e1WAAA='}
for relative_path, encoded in PAYLOAD.items():
    target = ROOT / relative_path
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_bytes(gzip.decompress(base64.b64decode(encoded)))
print('Prepared observed snapshot and code at', ROOT)


## Optional refresh from primary sources

Leave `REFRESH_PUBLIC_DATA=False` to use the audited snapshot. A failed download or changed release checksum stops acquisition; the script never generates substitute data.


In [ ]:
REFRESH_PUBLIC_DATA = False
if REFRESH_PUBLIC_DATA:
    subprocess.run([sys.executable, 'code/00_extract_public_data.py', '--wqp-year', '2020'], cwd=ROOT, check=True)


## Clean actual EPA records

Convert result and reporting-limit units independently. Keep non-detect intervals, unresolved stages and missing quarters. Raw, finished/entry and unresolved observations remain separate.


In [ ]:
def run_script(name):
    result = subprocess.run([sys.executable, 'code/' + name], cwd=ROOT, check=True, capture_output=True, text=True)
    print(result.stdout)
    if result.stderr: print(result.stderr)
run_script('01_clean_data.py')


## Describe observed coverage

The default analysis reports coverage and readiness. It does not substitute ambient WQP measurements for public-water observations or produce an unsupported data-center effect.


In [ ]:
run_script('02_analyze_data.py')
import pandas as pd
coverage = pd.read_csv(ROOT / 'results/observed_yearly_description.csv')
print(coverage.to_string(index=False))
try:
    from IPython.display import display, SVG
    display(SVG(filename=str(ROOT/'results/observed_data_coverage.svg')))
except ImportError:
    print(ROOT/'results/observed_data_coverage.svg')


## Verification checks

The artificial data inside the checks are software fixtures only. They are never appended to observed records.


In [ ]:
run_script('03_verify_pipeline.py')


## Forensic replication of the original generated results

This executes the supplied generator and estimator with output paths contained in the forensic directory. It reproduces the extreme-input challenge. Every generated result is retained under `SYNTHETIC_OUTPUTS`. These are not observed nitrate or weather records.


In [ ]:
RUN_FORENSIC_REPLICATION = True
if RUN_FORENSIC_REPLICATION:
    run_script('05_reproduce_original.py')
    checks = json.loads((ROOT/'forensic_reproduction/SYNTHETIC_OUTPUTS/reproduction_checks.json').read_text())
    assert checks['drop_file_changes_output'] is False
    assert checks['n_quarter_rows'] == 4176
    print(json.dumps(checks, indent=2))


## Requirements for an actual causal analysis

Review `data/required_inputs.json` and the accompanying manuscript. Verify facility-event dates, source/supplier connections, genuinely unexposed controls, measured weather and independent supply clusters. Blank event dates are not evidence of no exposure. The optional paired group-time function in `02_analyze_data.py` recomputes bootstrap uncertainty; it is not a complete doubly robust CS implementation. No causal estimates from verified Iowa inputs are presented here.


In [ ]:
print(json.dumps(json.loads((ROOT/'data/required_inputs.json').read_text()), indent=2))


## Save the reproduced files

This creates a ZIP with code, observed data and clearly separated forensic outputs. The large national cache is excluded. In Colab, use the file browser to download it.


In [ ]:
import zipfile
output = ROOT.parent / 'Iowa_Nitrate_Audit_Reproduced.zip'
with zipfile.ZipFile(output, 'w', zipfile.ZIP_DEFLATED) as z:
    for path in ROOT.rglob('*'):
        if path.is_file() and not {'cache','__pycache__'}.intersection(path.relative_to(ROOT).parts):
            z.write(path, path.relative_to(ROOT))
print(output)
